# **Importing Required Libraries**

In [ ]:
import re
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# modeling & pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# metrics
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    confusion_matrix, classification_report
)

# models
from xgboost import XGBRegressor, XGBClassifier

#visualisation
import matplotlib.pyplot as plt
import seaborn as sns

#Report Generation
!pip install reportlab
from reportlab.lib.pagesizes import letter, A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.colors import Color, HexColor
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.graphics.shapes import Drawing, Rect
from reportlab.graphics.charts.barcharts import VerticalBarChart
from reportlab.graphics.charts.piecharts import Pie
from reportlab.lib import colors
from datetime import datetime
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.3 MB/s eta 0:00:00


# **Loading the Dataset**

In [ ]:
df = pd.read_csv("IT_Hiring_Dataset.csv")
print(df.shape)
print(df.head(3))

(200, 18)
   Years_of_Experience Education_Level  Skill_Set_Score Job_Role_Level  \
0                    7         Diploma               54          Entry   
1                    6         Diploma               45            Mid   
2                    3         Diploma               49          Entry   

    Job_Type Work_Mode Location_Preference  Certifications_Count  \
0   Contract    Remote           Bangalore                     2   
1  Full-time    Hybrid           Hyderabad                     0   
2  Full-time   On-site              Remote                     2   

   Communication_Score  Problem_Solving_Score  Teamwork_Score  \
0                   36                     93              42   
1                   53                     43              81   
2                   59                     80              93   

   Leadership_Score  Cultural_Fit_Score  Assessment_Score  Expected_CTC  \
0                41                  54                95        700000   
1        

# **Skeleton Function for Shortlisting**

In [ ]:
def evaluate_candidate():
    edu_levels = {"HighSchool": 1, "Diploma": 2, "Bachelor": 3, "Master": 4, "PhD": 5}

    years_exp = int(input("Enter Years of Experience: "))
    education = input("Enter Education Level (Diploma/Bachelor/Master/PhD): ")
    if education not in edu_levels:
        print("Invalid education level. Please choose from Diploma, Bachelor, Master, or PhD.")
        return
    skill_score = int(input("Enter Skill Set Score (0-100): "))
    certifications = int(input("Enter Certifications Count: "))
    communication = int(input("Enter Communication Score (0-100): "))
    cultural_fit = int(input("Enter Cultural Fit Score (0-100): "))
    assessment = int(input("Enter Assessment Score (0-100): "))

    if (
        years_exp > 2 and
        edu_levels.get(education, 0) >= edu_levels["Bachelor"] and
        skill_score > 50 and
        certifications >= 2 and
        communication > 70 or
        cultural_fit > 60 or
        assessment > 70
    ):
        print("\n Candidate is Suitable for Hiring.")
    else:
        print("\n Candidate is Not Suitable for Hiring.")

evaluate_candidate()


Enter Years of Experience: 5
Enter Education Level (Diploma/Bachelor/Master/PhD): 90
Invalid education level. Please choose from Diploma, Bachelor, Master, or PhD.


In [ ]:
salary_col = None
for c in df.columns:
    if re.search(r"(salary|ctc|compensation|pay)", c, flags=re.I):
        if np.issubdtype(df[c].dropna().dtype, np.number):
            salary_col = c
            break
assert salary_col is not None, "Could not auto-detect a numeric salary column."
print("Detected salary column:", salary_col)


def derive_suitability(data: pd.DataFrame, salary_col: str) -> pd.Series:
    q90 = data[salary_col].quantile(0.90)
    conds = [
        data.get("Assessment_Score", pd.Series([0]*len(data))) >= 70,
        data.get("Cultural_Fit_Score", pd.Series([0]*len(data))) >= 60,
        data.get("Communication_Score", pd.Series([0]*len(data))) >= 50,
        data.get("Notice_Period_Days", pd.Series([999]*len(data))) <= 60,
        data[salary_col].fillna(data[salary_col].median()) <= q90,
    ]
    mask = np.logical_and.reduce(conds)
    return pd.Series(mask.astype(int), index=data.index, name="Suitable_for_Hire")

if "Suitable_for_Hire" not in df.columns:
    df["Suitable_for_Hire"] = derive_suitability(df, salary_col)

print("Suitability distribution (derived):")
print(df["Suitable_for_Hire"].value_counts())


Detected salary column: Expected_CTC
Suitability distribution (derived):
Suitable_for_Hire
0    176
1     24
Name: count, dtype: int64


# **Applying EDA**

In [ ]:
numeric_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
print("Numeric Columns:", numeric_cols)

# Plot correlation heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()

In [ ]:
cat_cols = ["Education_Level","Job_Role_Level","Job_Type","Work_Mode","Location_Preference"]
for col in cat_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(
        y=df[col],
        order=df[col].value_counts().index,
        palette="Set2"
    )
    plt.title(f"Category Counts: {col}", fontsize=14, fontweight="bold")
    plt.xlabel("Count")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


In [ ]:
#Boxplot with custom colors
plt.figure(figsize=(6,4))
sns.boxplot(
    x="Suitable_for_Hire",
    y="Assessment_Score",
    data=df,
    palette="Set2"
)
plt.title("Assessment Score vs Suitability", fontsize=14, fontweight="bold")
plt.xlabel("Suitable for Hire")
plt.ylabel("Assessment Score")
plt.tight_layout()
plt.show()

#Histogram with hue coloring
plt.figure(figsize=(6,4))
sns.histplot(
    data=df,
    x="Years_of_Experience",
    hue="Suitable_for_Hire",
    kde=True,
    element="step",
    palette="husl"
)
plt.title("Experience Distribution by Suitability", fontsize=14, fontweight="bold")
plt.xlabel("Years of Experience")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

#Pairplot with target hue
key_features = ["Years_of_Experience","Skill_Set_Score","Assessment_Score","Expected_CTC"]
sns.pairplot(
    df[key_features + ["Suitable_for_Hire"]],
    hue="Suitable_for_Hire",
    diag_kind="kde",
    palette="Set1"
)
plt.suptitle("Key Features Pairwise Relationships", y=1.02, fontsize=16, fontweight="bold")
plt.show()

# **Training the dataset over XGBoost Regressor Model**

In [ ]:
# separate feature columns (drop targets)
feature_cols = [c for c in df.columns if c not in {salary_col, "Suitable_for_Hire"}]
X_all = df[feature_cols].copy()

# identify column types
numeric_cols = [c for c in X_all.columns if np.issubdtype(X_all[c].dtype, np.number)]
object_cols  = [c for c in X_all.columns if X_all[c].dtype == "object"]

# split object columns into categorical vs free-text via a heuristic
free_text_cols = []
categorical_cols = []
for c in object_cols:
    s = X_all[c].astype(str).fillna("")
    avg_len = s.str.len().mean()
    nunique = X_all[c].nunique(dropna=True)
    unique_ratio = nunique / max(1, len(X_all))
    if (avg_len > 20) or (unique_ratio > 0.5):
        free_text_cols.append(c)
    else:
        categorical_cols.append(c)

print("numeric_cols     :", numeric_cols)
print("categorical_cols :", categorical_cols)
print("free_text_cols   :", free_text_cols, "(TF-IDF applied if any)")

# define transformers
num_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

cat_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# one TF-IDF per free-text column
text_transformers = []
for text_col in free_text_cols:
    text_transformers.append((f"tfidf__{text_col}", TfidfVectorizer(max_features=5000, ngram_range=(1,2)), text_col))

# ColumnTransformer
transformers = []
if numeric_cols:     transformers.append(("num", num_tf, numeric_cols))
if categorical_cols: transformers.append(("cat", cat_tf, categorical_cols))
transformers.extend(text_transformers)

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

numeric_cols     : ['Years_of_Experience', 'Skill_Set_Score', 'Certifications_Count', 'Communication_Score', 'Problem_Solving_Score', 'Teamwork_Score', 'Leadership_Score', 'Cultural_Fit_Score', 'Assessment_Score', 'Notice_Period_Days', 'Hiring_Decision']
categorical_cols : ['Education_Level', 'Job_Role_Level', 'Job_Type', 'Work_Mode', 'Location_Preference', 'Candidate_ID']
free_text_cols   : [] (TF-IDF applied if any)


In [ ]:
#XGBClassifier for suitability (1 = suitable, 0 = not)

y_cls = df["Suitable_for_Hire"].astype(int)
X_cls = X_all

Xtr_c, Xva_c, ytr_c, yva_c = train_test_split(X_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=42)

cls_pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.07,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    ))
])

cls_pipe.fit(Xtr_c, ytr_c)
proba = cls_pipe.predict_proba(Xva_c)[:, 1]
pred  = (proba >= 0.5).astype(int)

acc = accuracy_score(yva_c, pred)
r2   = r2_score(yva_c, pred)
prec, rec, f1, _ = precision_recall_fscore_support(yva_c, pred, average="binary", zero_division=0)

# ROC-AUC only if both classes present in yva_c
try:
    auc = roc_auc_score(yva_c, proba)
except ValueError:
    auc = float("nan")

#print(f"[Suitability] ACC={acc:.4f} | PREC={prec:.4f} | REC={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f}")
print("\nClassification report:\n", classification_report(yva_c, pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(yva_c, pred))
print("Accuracy: \n", acc*100)


Classification report:
               precision    recall  f1-score   support

           0     0.9722    1.0000    0.9859        35
           1     1.0000    0.8000    0.8889         5

    accuracy                         0.9750        40
   macro avg     0.9861    0.9000    0.9374        40
weighted avg     0.9757    0.9750    0.9738        40

Confusion matrix:
 [[35  0]
 [ 1  4]]
Accuracy: 
 97.5


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Hiring_Decision']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Hiring_Decision']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:
sample_reg = Xva_c.iloc[:5].copy()
sample_cls = Xva_c.iloc[:5].copy()

salary_preds = cls_pipe.predict(sample_reg)
suit_proba   = cls_pipe.predict_proba(sample_cls)[:, 1]
suit_pred    = (suit_proba >= 0.5).astype(int)

#print("\n--- Salary predictions (first 5 validation rows) ---")
out_sal = sample_reg.copy()
out_sal["Predicted_Salary"] = salary_preds
#print(out_sal.head(5))

print("\n--- Suitability predictions (first 5 validation rows) ---")
out_suit = sample_cls.copy()
out_suit["P(suitable)"] = suit_proba
out_suit["Predicted_Label"] = suit_pred
out_suit.head(5)



--- Suitability predictions (first 5 validation rows) ---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Hiring_Decision']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Hiring_Decision']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,Years_of_Experience,Education_Level,Skill_Set_Score,Job_Role_Level,Job_Type,Work_Mode,Location_Preference,Certifications_Count,Communication_Score,Problem_Solving_Score,Teamwork_Score,Leadership_Score,Cultural_Fit_Score,Assessment_Score,Notice_Period_Days,Hiring_Decision,Candidate_ID,P(suitable),Predicted_Label
67,2,PhD,64,Senior,Full-time,On-site,Pune,2,62,42,68,48,39,34,30,NaN,CD123,0.000453,0
172,5,Masters,54,Entry,Full-time,On-site,Chennai,3,53,89,55,27,45,39,90,NaN,CD133,0.001124,0
183,8,Bachelors,42,Mid,Internship,Remote,Pune,0,40,92,92,94,50,39,60,NaN,CD103,0.002892,0
2,3,Diploma,49,Entry,Full-time,On-site,Remote,2,59,80,93,100,82,48,15,NaN,CD123,0.042770,0
24,2,Diploma,99,Entry,Full-time,On-site,Pune,1,54,31,55,78,66,86,15,NaN,CD113,0.831668,1


# **Creating Gradio Interface**

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import warnings
import io
import os
import base64
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

class HRPredictionModel:
    def __init__(self):
        self.hiring_model = None
        self.salary_model = None
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.feature_columns = None
        self.dataset_info = None
        self.original_df = None  # Store original dataset for candidate search

    def load_and_preprocess_data(self, csv_file):
        print("Loading and preprocessing data...")

        # Read the uploaded CSV file
        df = pd.read_csv(csv_file.name)
        print(f"Successfully loaded file: {csv_file.name}")

        # Store original dataframe for candidate search
        self.original_df = df.copy()

        # Store dataset info for visualization
        self.dataset_info = {
            'shape': df.shape,
            'columns': list(df.columns),
            'missing_values': df.isnull().sum().to_dict(),
            'data_types': df.dtypes.to_dict(),
            'sample_data': df.head()
        }

        # Display basic info about the dataset
        print(f"Dataset shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")

        # Handle missing values in Prediction
        if 'Hiring_Decision' in df.columns:
            # Fill missing values with 'No'
            df['Hiring_Decision'] = df['Hiring_Decision'].fillna('No')
            # Clean the hiring decision column
            df['Hiring_Decision'] = df['Hiring_Decision'].astype(str).str.strip()
            # Check for various positive indicators
            positive_indicators = ['yes', 'hired', 'selected', 'accepted', '1', 'true']
            df['Hired'] = df['Hiring_Decision'].str.lower().isin(positive_indicators).astype(int)

            # If all are 0 or 1, create some realistic hiring decisions based on scores
            if df['Hired'].nunique() == 1:
                print("Warning: All candidates have the same hiring decision. Creating realistic hiring decisions based on scores...")

                # Create hiring decisions based on multiple criteria
                hiring_score = 0

                # Add points based on various factors
                if 'Assessment_Score' in df.columns:
                    hiring_score += (df['Assessment_Score'] >= 70) * 0.3
                if 'Skill_Set_Score' in df.columns:
                    hiring_score += (df['Skill_Set_Score'] >= 70) * 0.25
                if 'Communication_Score' in df.columns:
                    hiring_score += (df['Communication_Score'] >= 70) * 0.2
                if 'Years_of_Experience' in df.columns:
                    hiring_score += (df['Years_of_Experience'] >= 2) * 0.15
                if 'Cultural_Fit_Score' in df.columns:
                    hiring_score += (df['Cultural_Fit_Score'] >= 50) * 0.1

                # Add some randomness to make it realistic
                random_component = np.random.normal(0, 0.1, len(df))
                final_score = hiring_score + random_component

                # Top 30-40% get hired
                threshold = np.percentile(final_score, 65)
                df['Hired'] = (final_score >= threshold).astype(int)

        else:
            print("Warning: 'Hiring_Decision' column not found. Creating realistic hiring target based on candidate scores.")
            # Create realistic hiring decisions based on available scores
            hiring_score = np.random.normal(0.5, 0.2, len(df))
            df['Hired'] = (hiring_score > 0.5).astype(int)

        # Encode categorical variables
        categorical_cols = ['Education_Level', 'Job_Role_Level', 'Job_Type', 'Work_Mode', 'Location_Preference']

        for col in categorical_cols:
            if col in df.columns:
                le = LabelEncoder()
                # Handle missing values
                df[col] = df[col].fillna('Unknown')
                df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
                self.label_encoders[col] = le
                print(f"Encoded {col}: {list(le.classes_)}")

        # Define potential feature columns
        potential_features = [
            'Years_of_Experience', 'Education_Level_encoded', 'Skill_Set_Score',
            'Job_Role_Level_encoded', 'Job_Type_encoded', 'Work_Mode_encoded',
            'Location_Preference_encoded', 'Certifications_Count', 'Communication_Score',
            'Problem_Solving_Score', 'Teamwork_Score', 'Leadership_Score',
            'Cultural_Fit_Score', 'Assessment_Score', 'Notice_Period_Days'
        ]

        # Keep only available features
        available_features = []
        for col in potential_features:
            if col in df.columns:
                # Handle missing values in numeric columns
                if df[col].dtype in ['float64', 'int64']:
                    df[col] = df[col].fillna(df[col].median())
                available_features.append(col)

        self.feature_columns = available_features
        print(f"Available features ({len(available_features)}): {available_features}")

        # Prepare feature matrix and targets
        X = df[available_features]
        y_hiring = df['Hired']
        y_salary = df['Expected_CTC'] if 'Expected_CTC' in df.columns else pd.Series([50000] * len(df))

        print(f"Hiring distribution: {df['Hired'].value_counts().to_dict()}")
        if 'Expected_CTC' in df.columns:
            print(f"Salary range: ₹{y_salary.min():,.0f} - ₹{y_salary.max():,.0f}")

        return X, y_hiring, y_salary, df

    def train_models(self, X, y_hiring, y_salary):
        """Train both hiring decision and salary prediction models"""
        print("\nTraining models...")

        # Check if we have enough data
        if len(X) < 10:
            raise ValueError("Not enough data to train models (minimum 10 samples required)")

        # Check class distribution
        unique_classes = np.unique(y_hiring)
        print(f"Classes in target: {unique_classes}")
        print(f"Class distribution: {np.bincount(y_hiring)}")

        if len(unique_classes) < 2:
            raise ValueError(f"Need at least 2 classes for classification, got {len(unique_classes)}. Please check your hiring decisions.")

        # Split data - use stratified split only if we have both classes
        try:
            X_train, X_test, y_hiring_train, y_hiring_test, y_salary_train, y_salary_test = train_test_split(
                X, y_hiring, y_salary, test_size=0.2, random_state=42, stratify=y_hiring
            )
        except ValueError:
            # If stratified split fails, use regular split
            X_train, X_test, y_hiring_train, y_hiring_test, y_salary_train, y_salary_test = train_test_split(
                X, y_hiring, y_salary, test_size=0.2, random_state=42
            )

        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        # Train hiring decision model (Classification)
        print("Training hiring decision model...")

        # Use balanced class weights to handle imbalanced data
        self.hiring_model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric='logloss',
            scale_pos_weight=len(y_hiring_train[y_hiring_train == 0]) / len(y_hiring_train[y_hiring_train == 1]) if len(y_hiring_train[y_hiring_train == 1]) > 0 else 1
        )

        try:
            self.hiring_model.fit(X_train_scaled, y_hiring_train)
        except Exception as e:
            print(f"Error training hiring model: {e}")
            #print("Falling back to Random Forest classifier...")
            # Fallback: use a simple decision tree or logistic regression
            #from sklearn.ensemble import RandomForestClassifier
            #print("Falling back to Random Forest classifier...")
            #self.hiring_model = RandomForestClassifier(n_estimators=100, random_state=42)
            #self.hiring_model.fit(X_train_scaled, y_hiring_train)

        # Train salary prediction model (Regression) - only on hired candidates
        print("Training salary prediction model...")
        hired_mask_train = y_hiring_train == 1
        hired_mask_test = y_hiring_test == 1

        if hired_mask_train.sum() > 5:  # Need at least 5 hired candidates
            try:
                self.salary_model = xgb.XGBRegressor(
                    n_estimators=100,
                    max_depth=6,
                    learning_rate=0.1,
                    random_state=42
                )
                self.salary_model.fit(X_train_scaled[hired_mask_train], y_salary_train[hired_mask_train])
            except Exception as e:
                print(f"Error training salary model with XGBoost: {e}")
                # Fallback to Random Forest
                # sklearn.ensemble import RandomForestRegressor
                #print("Falling back to Random Forest regressor...")
                #self.salary_model = RandomForestRegressor(n_estimators=100, random_state=42)
                #self.salary_model.fit(X_train_scaled[hired_mask_train], y_salary_train[hired_mask_train])
        else:
            print("Warning: Not enough hired candidates to train salary model")
            self.salary_model = None

        # Evaluate models
        evaluation_results = self.evaluate_models(X_test_scaled, y_hiring_test, y_salary_test)

        return evaluation_results

    def evaluate_models(self, X_test, y_hiring_test, y_salary_test):
        """Evaluate both models and return results"""
        results = []

        # Hiring decision evaluation
        hiring_pred = self.hiring_model.predict(X_test)
        hiring_accuracy = accuracy_score(y_hiring_test, hiring_pred)

        results.append(f"🎯 HIRING DECISION MODEL:")
        results.append(f"Accuracy: {hiring_accuracy:.2f}")

        # Get classification report as string
        class_report = classification_report(y_hiring_test, hiring_pred, target_names=['Not Hired', 'Hired'])
        results.append("Classification Report:")
        results.append(class_report)

        # Salary prediction evaluation (only for hired candidates)
        hired_mask = y_hiring_test == 1
        if hired_mask.sum() > 0 and self.salary_model is not None:
            salary_pred = self.salary_model.predict(X_test[hired_mask])
            salary_mae = mean_absolute_error(y_salary_test[hired_mask], salary_pred)
            salary_r2 = r2_score(y_salary_test[hired_mask], salary_pred)

            results.append(f"\n💰 SALARY PREDICTION MODEL (for hired candidates):")
            #results.append(f"Mean Absolute Error: ₹{salary_mae:,.0f}")
            #results.append(f"R² Score: {salary_r2:.3f}")

        # Feature importance
        feature_importance = self.get_feature_importance()
        results.extend(feature_importance)

        return "\n".join(results)

    def get_feature_importance(self):
        """Get feature importance for both models"""
        results = []
        results.append(f"\n🔍 FEATURE IMPORTANCE:")

        # Hiring model importance
        if hasattr(self.hiring_model, 'feature_importances_'):
            hiring_importance = self.hiring_model.feature_importances_
            hiring_features = list(zip(self.feature_columns, hiring_importance))
            hiring_features.sort(key=lambda x: x[1], reverse=True)

            results.append(f"\nTop 5 features for Hiring Decision:")
            for i, (feature, importance) in enumerate(hiring_features[:5], 1):
                results.append(f"{i}. {feature}: {importance:.1f}")

        # Salary model importance
        if self.salary_model is not None and hasattr(self.salary_model, 'feature_importances_'):
            salary_importance = self.salary_model.feature_importances_
            salary_features = list(zip(self.feature_columns, salary_importance))
            salary_features.sort(key=lambda x: x[1], reverse=True)

            results.append(f"\nTop 5 features for Salary Prediction:")
            for i, (feature, importance) in enumerate(salary_features[:5], 1):
                results.append(f"{i}. {feature}: {importance:.1f}")

        return results

    def search_candidates(self, education_filter=None, location_filter=None):
        """Search candidates from the dataset and predict their hiring suitability"""
        if self.original_df is None:
            return "No dataset loaded. Please train the model first."

        if self.hiring_model is None:
            return "Model not trained. Please train the model first."

        df = self.original_df.copy()

        # Apply filters
        if education_filter and education_filter != "All":
            df = df[df['Education_Level'] == education_filter]

        if location_filter and location_filter != "All":
            df = df[df['Location_Preference'] == location_filter]

        if len(df) == 0:
            return "No candidates found matching the criteria."

        results = []
        suitable_candidates = []

        for idx, row in df.iterrows():
            # Prepare candidate data for prediction
            candidate_data = {
                'Years_of_Experience': row.get('Years_of_Experience', 0),
                'Education_Level': row.get('Education_Level', 'Bachelor'),
                'Skill_Set_Score': row.get('Skill_Set_Score', 0),
                'Job_Role_Level': row.get('Job_Role_Level', 'Entry'),
                'Job_Type': row.get('Job_Type', 'Full-time'),
                'Work_Mode': row.get('Work_Mode', 'Office'),
                'Location_Preference': row.get('Location_Preference', 'Bangalore'),
                'Certifications_Count': row.get('Certifications_Count', 0),
                'Communication_Score': row.get('Communication_Score', 0),
                'Problem_Solving_Score': row.get('Problem_Solving_Score', 0),
                'Teamwork_Score': row.get('Teamwork_Score', 0),
                'Leadership_Score': row.get('Leadership_Score', 0),
                'Cultural_Fit_Score': row.get('Cultural_Fit_Score', 0),
                'Assessment_Score': row.get('Assessment_Score', 0),
                'Notice_Period_Days': row.get('Notice_Period_Days', 30)
            }

            try:
                # Get prediction
                prediction_result, hiring_prob, hiring_decision, predicted_salary = self.predict_candidate(candidate_data)

                candidate_id = row.get('Candidate_ID', f'CAND_{idx}')
                email_id = row.get('E_Mail_ID', f'candidate{idx}@example.com')

                if hiring_decision == 1:  # Suitable for hiring
                    suitable_candidates.append({
                        'Candidate_ID': candidate_id,
                        'E_Mail_ID': email_id,
                        'Years_of_Experience': candidate_data['Years_of_Experience'],
                        'Education_Level': candidate_data['Education_Level'],
                        'Skill_Set_Score': candidate_data['Skill_Set_Score'],
                        'Communication_Score': candidate_data['Communication_Score'],
                        'Cultural_Fit_Score': candidate_data['Cultural_Fit_Score'],
                        'Assessment_Score': candidate_data['Assessment_Score'],
                        'Expected_CTC': row.get('Expected_CTC', 'N/A'),
                        'Hiring_Probability': hiring_prob,
                        'Predicted_Salary': predicted_salary if predicted_salary else 'N/A'
                    })

                    results.append(f"✅ {candidate_id} | {email_id} | SUITABLE (Probability: {hiring_prob:.2f})")
                else:
                    results.append(f"❌ {candidate_id} | {email_id} | NOT SUITABLE (Probability: {hiring_prob:.2f})")

            except Exception as e:
                results.append(f"⚠️ {candidate_id} | Error in prediction: {str(e)}")

        # Store suitable candidates for report generation
        self.suitable_candidates = suitable_candidates

        summary = f"\n📊 SEARCH SUMMARY:\n"
        summary += f"Total Candidates Found: {len(df)}\n"
        summary += f"Suitable for Hiring: {len(suitable_candidates)}\n"
        summary += f"Success Rate: {len(suitable_candidates)/len(df)*100:.1f}%\n"

        return summary + "\n" + "\n".join(results)

    def predict_candidate(self, candidate_data):
        """Predict hiring decision and salary for a single candidate"""
        if self.hiring_model is None:
            raise ValueError("Model not trained. Please train the model first.")

        # Prepare input data
        input_df = pd.DataFrame([candidate_data])

        # Encode categorical variables using fitted label encoders
        for col, le in self.label_encoders.items():
            if col in input_df.columns:
                # Handle unknown categories by using the most frequent class
                try:
                    input_df[col + '_encoded'] = le.transform(input_df[col].astype(str))
                except ValueError:
                    # If category not seen during training, use the first class
                    print(f"Warning: Unknown category '{input_df[col].iloc[0]}' for {col}. Using default.")
                    input_df[col + '_encoded'] = 0

        # Select and order features
        X_input = pd.DataFrame()
        for col in self.feature_columns:
            if col in input_df.columns:
                X_input[col] = input_df[col]
            else:
                X_input[col] = [0]  # Default value

        X_input_scaled = self.scaler.transform(X_input)

        # Predict hiring decision
        hiring_prob = self.hiring_model.predict_proba(X_input_scaled)[0][1]
        hiring_decision = self.hiring_model.predict(X_input_scaled)[0]

        results = []
        results.append("🎯 AI MODEL PREDICTION:")
        results.append(f"Hiring Probability: {hiring_prob:.2f}")
        results.append(f"Hiring Decision: {'✅ HIRED' if hiring_decision == 1 else '❌ NOT HIRED'}")

        predicted_salary = None
        # Predict salary if hired
        if hiring_decision == 1 and self.salary_model is not None:
            predicted_salary = self.salary_model.predict(X_input_scaled)[0]
            results.append(f"Predicted Salary: ₹{predicted_salary:,.0f}")
        elif hiring_decision == 1:
            results.append("Salary prediction model not available")

        return "\n".join(results), hiring_prob, hiring_decision, predicted_salary

    def create_dataset_visualizations(self):
        """Create visualizations for the dataset"""
        if self.dataset_info is None:
            return None, "No dataset loaded for visualization"

        # Create multiple visualizations
        figures = []

        try:
            # Read the dataset again for visualization
            # This is a simplified approach - in production, you'd want to store the dataframe
            return "Dataset visualizations will be available after training", ""
        except Exception as e:
            return None, f"Error creating visualizations: {str(e)}"

# Enhanced report generation functions
def generate_candidates_report(suitable_candidates):
    """Generate a PDF report for suitable candidates"""
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.colors import HexColor
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
    from reportlab.lib import colors

    if not suitable_candidates:
        return "No suitable candidates found to generate report."

    # Create filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = "Suitable_Candidates_Report.pdf"

    # Create document
    doc = SimpleDocTemplate(filename, pagesize=A4, topMargin=0.5*inch, bottomMargin=0.5*inch)

    # Styles
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('Title', parent=styles['Title'], fontSize=20,
                                textColor=HexColor('#2E4A6B'), alignment=TA_CENTER)
    heading_style = ParagraphStyle('Heading', parent=styles['Heading1'], fontSize=14,
                                  textColor=HexColor('#34495E'), spaceAfter=12)
    normal_style = ParagraphStyle('Normal', parent=styles['Normal'], fontSize=10,
                                 textColor=HexColor('#2C3E50'))

    # Build content
    story = []

    # Title
    story.append(Paragraph("SUITABLE CANDIDATES REPORT", title_style))
    story.append(Spacer(1, 0.3*inch))

    # Report info
    report_date = datetime.now().strftime("%B %d, %Y at %I:%M %p")
    story.append(Paragraph(f"Report Generated: {report_date}", normal_style))
    story.append(Paragraph(f"Total Suitable Candidates: {len(suitable_candidates)}", normal_style))
    story.append(Spacer(1, 0.2*inch))

    # Candidates details
    for i, candidate in enumerate(suitable_candidates, 1):
        story.append(Paragraph(f"CANDIDATE No.{i}", heading_style))

        candidate_data = [
            ['Attribute', 'Value'],
            ['Candidate ID', candidate.get('Candidate_ID', 'N/A')],
            ['Email', candidate.get('E_Mail_ID', 'N/A')],
            ['Experience', f"{candidate.get('Years_of_Experience', 'N/A')} years"],
            ['Education', candidate.get('Education_Level', 'N/A')],
            ['Technical Skills', f"{candidate.get('Skill_Set_Score', 'N/A')}/100"],
            ['Communication', f"{candidate.get('Communication_Score', 'N/A')}/100"],
            ['Cultural Fit', f"{candidate.get('Cultural_Fit_Score', 'N/A')}/100"],
            ['Assessment Score', f"{candidate.get('Assessment_Score', 'N/A')}/100"],
            ['Expected CTC', f"₹{candidate.get('Expected_CTC', 'N/A'):,}" if candidate.get('Expected_CTC') != 'N/A' else 'N/A'],
            ['Hiring Probability', f"{candidate.get('Hiring_Probability', 0):.2f}"],
            ['Predicted Salary', f"₹{candidate.get('Predicted_Salary', 'N/A'):,.0f}" if candidate.get('Predicted_Salary') != 'N/A' else 'N/A'],
        ]

        candidate_table = Table(candidate_data, colWidths=[2*inch, 2.5*inch])
        candidate_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), HexColor('#3498DB')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 10),
            ('BACKGROUND', (0, 1), (-1, -1), HexColor('#F8F9FA')),
            ('GRID', (0, 0), (-1, -1), 1, HexColor('#BDC3C7')),
        ]))

        story.append(candidate_table)
        story.append(Spacer(1, 0.2*inch))

    # Build PDF
    doc.build(story)
    return filename

def generate_pdf_report(candidate_data, rule_result, ai_result=None):
    """Generate a comprehensive PDF report"""
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.colors import HexColor
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
    from reportlab.lib import colors

    # Create filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = "HR_Report.pdf"

    # Create document
    doc = SimpleDocTemplate(filename, pagesize=A4, topMargin=0.5*inch, bottomMargin=0.5*inch)

    # Styles
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('Title', parent=styles['Title'], fontSize=20,
                                textColor=HexColor('#2E4A6B'), alignment=TA_CENTER)
    heading_style = ParagraphStyle('Heading', parent=styles['Heading1'], fontSize=14,
                                  textColor=HexColor('#34495E'), spaceAfter=12)
    normal_style = ParagraphStyle('Normal', parent=styles['Normal'], fontSize=10,
                                 textColor=HexColor('#2C3E50'))

    # Build content
    story = []

    # Title
    story.append(Paragraph("HR EVALUATION REPORT", title_style))
    story.append(Spacer(1, 0.3*inch))

    # Report info
    report_date = datetime.now().strftime("%B %d, %Y at %I:%M %p")
    story.append(Paragraph(f"Report Generated: {report_date}", normal_style))
    story.append(Spacer(1, 0.2*inch))

    # Candidate Profile
    story.append(Paragraph("CANDIDATE PROFILE", heading_style))

    profile_data = [
        ['Attribute', 'Value'],
        ['Years of Experience', f"{candidate_data.get('years_experience', 'N/A')} years"],
        ['Education Level', candidate_data.get('education', 'N/A')],
        ['Technical Skills', f"{candidate_data.get('skill_score', 'N/A')}/100"],
        ['Communication Score', f"{candidate_data.get('communication', 'N/A')}/100"],
        ['Cultural Fit Score', f"{candidate_data.get('cultural_fit', 'N/A')}/100"],
        ['Assessment Score', f"{candidate_data.get('assessment', 'N/A')}/100"],
        ['Certifications', str(candidate_data.get('certifications', 'N/A'))],
    ]

    profile_table = Table(profile_data, colWidths=[2*inch, 2*inch])
    profile_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), HexColor('#3498DB')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 10),
        ('BACKGROUND', (0, 1), (-1, -1), HexColor('#F8F9FA')),
        ('GRID', (0, 0), (-1, -1), 1, HexColor('#BDC3C7')),
    ]))

    story.append(profile_table)
    story.append(Spacer(1, 0.2*inch))

    # Rule-based results
    story.append(Paragraph("RULE-BASED EVALUATION", heading_style))
    story.append(Paragraph(rule_result, normal_style))
    story.append(Spacer(1, 0.2*inch))

    # AI results if available
    if ai_result:
        story.append(Paragraph("AI MODEL PREDICTION", heading_style))
        story.append(Paragraph(ai_result, normal_style))

    # Build PDF
    doc.build(story)

    return filename

def get_rating(score):
    """Convert numeric score to rating"""
    if score >= 90:
        return "Excellent"
    elif score >= 80:
        return "Very Good"
    elif score >= 70:
        return "Good"
    elif score >= 60:
        return "Average"
    elif score >= 50:
        return "Below Average"
    else:
        return "Poor"

# Rule-based evaluation function
def evaluate_candidate_rule_based(years_exp, education, skill_score, certifications,
                                communication, cultural_fit, assessment):
    """Rule-based evaluation function"""
    edu_levels = {"HighSchool": 1, "Diploma": 2,"Bachelors": 3, "Masters": 4, "PhD": 5}

    # Map education input to standard format
    education_score = edu_levels.get(education, 0)

    criteria_met = []
    criteria_failed = []

    # Check each criterion
    if years_exp > 2:
        criteria_met.append(f"✅ Experience: {years_exp} years (Required: >2)")
    else:
        criteria_failed.append(f"❌ Experience: {years_exp} years (Required: >2)")

    if education_score >= edu_levels["Bachelors"]:
        criteria_met.append(f"✅ Education: {education} (Required: Bachelors+)")
    else:
        criteria_failed.append(f"❌ Education: {education} (Required: Bachelors+)")

    if skill_score > 50:
        criteria_met.append(f"✅ Skills: {skill_score}/100 (Required: >50)")
    else:
        criteria_failed.append(f"❌ Skills: {skill_score}/100 (Required: >50)")

    if certifications >= 2:
        criteria_met.append(f"✅ Certifications: {certifications} (Required: ≥2)")
    else:
        criteria_failed.append(f"❌ Certifications: {certifications} (Required: ≥2)")

    if communication > 70:
        criteria_met.append(f"✅ Communication: {communication}/100 (Required: >70)")
    else:
        criteria_failed.append(f"❌ Communication: {communication}/100 (Required: >70)")

    if cultural_fit > 60:
        criteria_met.append(f"✅ Cultural Fit: {cultural_fit}/100 (Required: >60)")
    else:
        criteria_failed.append(f"❌ Cultural Fit: {cultural_fit}/100 (Required: >60)")

    if assessment > 70:
        criteria_met.append(f"✅ Assessment: {assessment}/100 (Required: >70)")
    else:
        criteria_failed.append(f"❌ Assessment: {assessment}/100 (Required: >70)")

    # Final decision
    all_criteria_met = (
        years_exp > 2 and
        education_score >= edu_levels["Bachelor"] and
        skill_score > 50 and
        certifications >= 2 and
        communication > 60 or
        cultural_fit > 60 or
        assessment > 70
    )

    results = []
    results.append("📋 RULE-BASED EVALUATION:")
    results.append(f"\n📊 CRITERIA ANALYSIS:")

    for criterion in criteria_met:
        results.append(criterion)

    for criterion in criteria_failed:
        results.append(criterion)

    results.append(f"\n🎯 FINAL DECISION:")
    if all_criteria_met:
        results.append("✅ CANDIDATE IS SUITABLE FOR HIRING")
        results.append(f"📈 Criteria Met: {len(criteria_met)}/{len(criteria_met) + len(criteria_failed)}")
    else:
        results.append("❌ CANDIDATE IS NOT SUITABLE FOR HIRING")
        results.append(f"📉 Criteria Met: {len(criteria_met)}/{len(criteria_met) + len(criteria_failed)}")

    return "\n".join(results)

# Global model instance
hr_model = HRPredictionModel()

def create_dataset_visualizations(csv_file):
    """Create visualizations for uploaded dataset"""
    if csv_file is None:
        return None, None, None, "Please upload a dataset first"

    try:
        df = pd.read_csv(csv_file.name)

        # Basic statistics
        stats_html = f"""
        <div style='background: #ffffff; padding: 20px; border-radius: 10px; margin: 10px 0; border: 1px solid #ddd;'>
            <h3 style='color: #2c3e50; margin-top: 0;'>📊 Dataset Overview</h3>
            <p style='color: #2c3e50;'><strong>Shape:</strong> {df.shape[0]} rows × {df.shape[1]} columns</p>
            <p style='color: #2c3e50;'><strong>Memory Usage:</strong> {df.memory_usage(deep=True).sum() / 1024:.2f} KB</p>
            <p style='color: #2c3e50;'><strong>Missing Values:</strong> {df.isnull().sum().sum()} total</p>
        </div>
        """

        #--------------------------------------------
        # Necessary Visualisation
        # -------------------------------------------

        # Create visualizations using matplotlib for better compatibility
        fig1, axes1 = plt.subplots(2, 2, figsize=(15, 12))
        fig1.suptitle('Dataset Analysis Dashboard', fontsize=16, fontweight='bold')

        # 1. Missing values heatmap
        if df.isnull().sum().sum() > 0:
            sns.heatmap(df.isnull(), cbar=True, ax=axes1[0,0], cmap='viridis')
            axes1[0,0].set_title('Missing Values Pattern')
        else:
            axes1[0,0].text(0.5, 0.5, 'No Missing Values', ha='center', va='center',
                           transform=axes1[0,0].transAxes, fontsize=12)
            axes1[0,0].set_title('Missing Values Pattern')

        # 2. Numeric columns distribution
        numeric_cols = df.select_dtypes(include=[np.number]).columns[:4]
        if len(numeric_cols) > 0:
            df[numeric_cols].hist(ax=axes1[0,1], bins=20, alpha=0.7)
            axes1[0,1].set_title('Numeric Features Distribution')
        else:
            axes1[0,1].text(0.5, 0.5, 'No Numeric Columns', ha='center', va='center',
                           transform=axes1[0,1].transAxes)
            axes1[0,1].set_title('Numeric Features Distribution')

        # 3. Categorical columns
        categorical_cols = df.select_dtypes(include=['object']).columns[:3]
        if len(categorical_cols) > 0:
            col = categorical_cols[0]
            value_counts = df[col].value_counts().head(10)
            axes1[1,0].bar(range(len(value_counts)), value_counts.values)
            axes1[1,0].set_xticks(range(len(value_counts)))
            axes1[1,0].set_xticklabels(value_counts.index, rotation=45)
            axes1[1,0].set_title(f'Distribution of {col}')
        else:
            axes1[1,0].text(0.5, 0.5, 'No Categorical Columns', ha='center', va='center',
                           transform=axes1[1,0].transAxes)
            axes1[1,0].set_title('Categorical Distribution')

        # 4. Correlation heatmap
        numeric_df = df.select_dtypes(include=[np.number])
        if len(numeric_df.columns) > 1:
            corr = numeric_df.corr()
            sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes1[1,1])
            axes1[1,1].set_title('Feature Correlations')
        else:
            axes1[1,1].text(0.5, 0.5, 'Insufficient Numeric Data', ha='center', va='center',
                           transform=axes1[1,1].transAxes)
            axes1[1,1].set_title('Feature Correlations')

        plt.tight_layout()

        # Create second figure for hiring analysis if hiring data exists
        fig2, axes2 = plt.subplots(2, 2, figsize=(15, 12))
        fig2.suptitle('HR Analytics Dashboard', fontsize=16, fontweight='bold')

        # Check for hiring decision column
        hiring_col = None
        for col in ['Hiring_Decision', 'hired', 'Hired', 'decision']:
            if col in df.columns:
                hiring_col = col
                break

        if hiring_col:
            # Hiring decision distribution
            hiring_counts = df[hiring_col].value_counts()
            axes2[0,0].pie(hiring_counts.values, labels=hiring_counts.index, autopct='%1.1f%%')
            axes2[0,0].set_title('Hiring Decision Distribution')
        else:
            axes2[0,0].text(0.5, 0.5, 'No Hiring Decision Data', ha='center', va='center',
                           transform=axes2[0,0].transAxes)
            axes2[0,0].set_title('Hiring Decision Distribution')

        # Experience vs Skills (if available)
        exp_col = None
        skill_col = None
        for col in df.columns:
            if 'experience' in col.lower() or 'exp' in col.lower():
                exp_col = col
            if 'skill' in col.lower() and df[col].dtype in ['int64', 'float64']:
                skill_col = col

        if exp_col and skill_col:
            axes2[0,1].scatter(df[exp_col], df[skill_col], alpha=0.6)
            axes2[0,1].set_xlabel(exp_col)
            axes2[0,1].set_ylabel(skill_col)
            axes2[0,1].set_title('Experience vs Skills')
        else:
            axes2[0,1].text(0.5, 0.5, 'Experience/Skills Data Not Available', ha='center', va='center',
                           transform=axes2[0,1].transAxes)
            axes2[0,1].set_title('Experience vs Skills')

        # Salary distribution (if available)
        salary_col = None
        for col in df.columns:
            if 'salary' in col.lower() or 'ctc' in col.lower() or 'compensation' in col.lower():
                if df[col].dtype in ['int64', 'float64']:
                    salary_col = col
                    break

        if salary_col:
            axes2[1,0].hist(df[salary_col].dropna(), bins=20, alpha=0.7, color='green')
            axes2[1,0].set_xlabel(salary_col)
            axes2[1,0].set_ylabel('Frequency')
            axes2[1,0].set_title('Salary Distribution')
        else:
            axes2[1,0].text(0.5, 0.5, 'No Salary Data Available', ha='center', va='center',
                           transform=axes2[1,0].transAxes)
            axes2[1,0].set_title('Salary Distribution')

        # Education level distribution (if available)
        edu_col = None
        for col in df.columns:
            if 'education' in col.lower() or 'degree' in col.lower():
                edu_col = col
                break

        if edu_col:
            edu_counts = df[edu_col].value_counts()
            axes2[1,1].bar(range(len(edu_counts)), edu_counts.values)
            axes2[1,1].set_xticks(range(len(edu_counts)))
            axes2[1,1].set_xticklabels(edu_counts.index, rotation=45)
            axes2[1,1].set_title('Education Level Distribution')
        else:
            axes2[1,1].text(0.5, 0.5, 'No Education Data Available', ha='center', va='center',
                           transform=axes2[1,1].transAxes)
            axes2[1,1].set_title('Education Level Distribution')

        plt.tight_layout()

        return fig1, fig2, stats_html, "Dataset visualizations created successfully!"

    except Exception as e:
        return None, None, None, f"Error creating visualizations: {str(e)}"


# --------------------------------------------------
# Uploading the Dataset for the training in the interface
# --------------------------------------------------

def train_model(csv_file):
    """Function to train the model with uploaded CSV"""
    if csv_file is None:
        return "⚠️ Please upload a CSV file first.", None, None, None

    try:
        global hr_model
        hr_model = HRPredictionModel()  # Reset model

        # Load and preprocess data
        X, y_hiring, y_salary, df = hr_model.load_and_preprocess_data(csv_file)

        # Create visualizations
        fig1, fig2, stats_html, viz_message = create_dataset_visualizations(csv_file)

        # Train models
        evaluation_results = hr_model.train_models(X, y_hiring, y_salary)

        training_message = f"✅ Model training completed successfully!\n\n{evaluation_results}"

        return training_message, fig1, fig2, stats_html

    except Exception as e:
        return f"❌ Error training model: {str(e)}", None, None, None

def search_candidates_by_criteria(education_filter, location_filter):
    """Search candidates based on education and location criteria"""
    global hr_model

    if hr_model.hiring_model is None:
        return "⚠️ Please train the AI model first by uploading a dataset in the 'AI Model Training' tab."

    search_result = hr_model.search_candidates(education_filter, location_filter)
    return search_result

def download_suitable_candidates_report():
    """Generate and return downloadable file for suitable candidates report"""
    global hr_model

    if not hasattr(hr_model, 'suitable_candidates') or not hr_model.suitable_candidates:
        return None, "No suitable candidates found. Please search for candidates first."

    try:
        filename = generate_candidates_report(hr_model.suitable_candidates)
        return filename, f"✅ Report generated successfully! Click to download: {filename}"
    except Exception as e:
        return None, f"❌ Error generating report: {str(e)}"

def predict_comprehensive(years_exp, education, skill_score, job_level, job_type,
                         work_mode, location, certifications, communication,
                         problem_solving, teamwork, leadership, cultural_fit,
                         assessment, notice_period):
    """Comprehensive prediction using both rule-based and ML approaches"""

    # Rule-based evaluation
    rule_result = evaluate_candidate_rule_based(
        years_exp, education, skill_score, certifications,
        communication, cultural_fit, assessment
    )

    # ML prediction (if model is trained)
    ml_result = "🤖 AI MODEL: Not trained yet. Please train the model first."
    hiring_prob = 0
    hiring_decision = 0
    predicted_salary = None

    if hr_model.hiring_model is not None:
        try:
            candidate_data = {
                'Years_of_Experience': years_exp,
                'Education_Level': education,
                'Skill_Set_Score': skill_score,
                'Job_Role_Level': job_level,
                'Job_Type': job_type,
                'Work_Mode': work_mode,
                'Location_Preference': location,
                'Certifications_Count': certifications,
                'Communication_Score': communication,
                'Problem_Solving_Score': problem_solving,
                'Teamwork_Score': teamwork,
                'Leadership_Score': leadership,
                'Cultural_Fit_Score': cultural_fit,
                'Assessment_Score': assessment,
                'Notice_Period_Days': notice_period
            }
            ml_result, hiring_prob, hiring_decision, predicted_salary = hr_model.predict_candidate(candidate_data)
        except Exception as e:
            ml_result = f"🤖 AI MODEL: Error making prediction: {str(e)}"

    # Combine results
    combined_result = f"""
{rule_result}

{'='*60}

{ml_result}

{'='*60}

💡 RECOMMENDATION SUMMARY:
The system provides two complementary approaches:
1. Rule-based evaluation uses predefined criteria for quick screening
2. AI model learns from historical data for nuanced predictions

For best results, consider both evaluations in your hiring decision.
    """

    # Create download link for report
    try:
        report_data = {
            'years_experience': years_exp,
            'education': education,
            'skill_score': skill_score,
            'job_level': job_level,
            'job_type': job_type,
            'work_mode': work_mode,
            'location': location,
            'certifications': certifications,
            'communication': communication,
            'problem_solving': problem_solving,
            'teamwork': teamwork,
            'leadership': leadership,
            'cultural_fit': cultural_fit,
            'assessment': assessment,
            'notice_period': notice_period
        }

        report_filename = generate_pdf_report(report_data, rule_result, ml_result)

        return combined_result.strip(), report_filename

    except Exception as e:
        return combined_result.strip(), f"Error generating report: {str(e)}"

# ----------------------------------------------------------
# CSS Integration
# ----------------------------------------------------------

def create_interface():
    """Create the enhanced Gradio interface"""

    # Custom CSS for better text visibility and modern design
    custom_css = """
    .gradio-container {
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        min-height: 100vh;
    }
    .gr-button-primary {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
        border: none !important;
        color: white !important;
        font-weight: bold !important;
        transition: all 0.3s ease !important;
    }
    .gr-button-primary:hover {
        transform: translateY(-2px) !important;
        box-shadow: 0 8px 25px rgba(0,0,0,0.3) !important;
    }
    .gr-textbox textarea {
        background-color: #ffffff !important;
        color: #2c3e50 !important;
        border: 1px solid #bdc3c7 !important;
    }
    .gr-textbox label {
        color: #2c3e50 !important;
        font-weight: 600 !important;
    }
    .info-box {
        background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
        padding: 20px;
        border-radius: 12px;
        margin: 15px 0;
        color: white;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }
    .metric-box {
        background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
        padding: 20px;
        border-radius: 12px;
        margin: 15px 0;
        color: white;
        text-align: center;
        font-weight: bold;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }
    .section-header {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 15px;
        border-radius: 8px;
        margin: 20px 0 10px 0;
        text-align: center;
        font-weight: bold;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }
    .gr-tab-nav {
        background: rgba(255,255,255,0.9) !important;
    }
    .gr-tab-nav button {
        color: #2c3e50 !important;
        font-weight: 600 !important;
    }
    .gr-tab-nav button.selected {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
        color: white !important;
    }
    .main-container {
        background: rgba(255,255,255,0.95);
        border-radius: 15px;
        padding: 20px;
        margin: 20px;
        box-shadow: 0 10px 30px rgba(0,0,0,0.1);
    }
    .gr-slider label, .gr-dropdown label, .gr-file label {
        color: #2c3e50 !important;
        font-weight: 600 !important;
    }
    .gr-html {
        color: #7ea86f !important;
    }
    .gr-html h1, .gr-html h2, .gr-html h3, .gr-html h4, .gr-html h5, .gr-html h6 {
        color: #2c3e50 !important;
    }
    .gr-html p, .gr-html li, .gr-html span, .gr-html div {
        color: #2c3e50 !important;
    }
    .gr-html strong, .gr-html b {
        color: #2c3e50 !important;
    }
    .gr-html ul, .gr-html ol {
        color: #2c3e50 !important;
    }
    .gr-html table {
        color: #2c3e50 !important;
    }
    .gr-html table th, .gr-html table td {
        color: #2c3e50 !important;
    }
        /* White text for specific textboxes */
    .white-text textarea, .white-text input {
        background-color: #2c3e50 !important;
        color: #ffffff !important;
        border: 1px solid #34495e !important;
    }

    /* White text for dropdowns */
    .white-dropdown select {
        background-color: #2c3e50 !important;
        color: #ffffff !important;
        border: 1px solid #34495e !important;
    }
    /* Force text visibility in all contexts */
    * {
        color: #2c3e50 !important;
    }
    """

    with gr.Blocks(
        title="🚀 Advanced HR Prediction System",
        theme=gr.themes.Soft(
            primary_hue="blue",
            secondary_hue="purple",
            neutral_hue="slate"
        ),
        css=custom_css
    ) as demo:

        # Main container wrapper
        with gr.Column(elem_classes=["main-container"]):

            # Header section
            gr.HTML("""
            <div style="text-align: center; padding: 40px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 15px; margin-bottom: 30px; box-shadow: 0 10px 30px rgba(0,0,0,0.2);">
                <h1 style="font-size: 3.5em; margin: 0; text-shadow: 2px 2px 4px rgba(0,0,0,0.3); font-weight: 700; color: white !important;">🚀 HR Prediction System</h1>
                <h2 style="font-size: 1.8em; margin: 15px 0; opacity: 0.95; font-weight: 400; color: white !important;">AI-Powered Hiring Decisions & Salary Predictions</h2>
                <p style="font-size: 1.2em; margin: 0; opacity: 0.85; color: white !important;">Combining Rule-Based Logic with Advanced Machine Learning</p>
            </div>
            """)

            # Main tabs with enhanced styling
            with gr.Tabs() as main_tabs:

                # AI Model Training Tab (moved to first)
                with gr.Tab("📊 AI Model Training", elem_id="training-tab"):
                    gr.HTML("""
                    <div class="info-box">
                        <h3 style="margin: 0 0 10px 0; color: white !important;">📊 Train Advanced AI Models</h3>
                        <p style="margin: 0; color: white !important;">Upload your historical HR data to train sophisticated machine learning models for hiring decisions and salary predictions. Get instant dataset analytics and visualizations.</p>
                    </div>
                    """)

                    with gr.Row():
                        with gr.Column(scale=1):
                            gr.HTML("""
                            <div class="metric-box">
                                <h3 style="margin: 0 0 15px 0; color: white !important;">📋 Data Requirements</h3>
                                <ul style="margin: 10px 0; text-align: left; color: white !important;">
                                    <li style="color: white !important;">CSV format with candidate information</li>
                                    <li style="color: white !important;">Hiring decisions (Yes/No, Hired/Not Hired)</li>
                                    <li style="color: white !important;">Salary/CTC information</li>
                                    <li style="color: white !important;">Skills, experience, and assessment scores</li>
                                    <li style="color: white !important;">Minimum 50+ records for best results</li>
                                </ul>
                            </div>
                            """)

                            csv_upload = gr.File(
                                label="📁 Upload HR Dataset (CSV)",
                                file_types=[".csv"],
                                type="filepath",
                                height=120,
                                elem_classes=["white-text"]
                            )

                            train_btn = gr.Button(
                                "🚀 Train AI Models & Generate Analytics",
                                variant="primary",
                                size="lg",
                                elem_classes=["white-text"]
                            )

                        with gr.Column(scale=2):
                            training_output = gr.Textbox(
                                label="🔬 Model Training Results & Analytics",
                                lines=15,
                                max_lines=25,
                                interactive=False,
                                placeholder="Training results and model performance metrics will appear here...",
                                show_copy_button=True,
                                elem_classes=["white-text"]
                            )

                    # Dataset Analytics Section
                    gr.HTML('<div class="section-header"><span style="color: white !important;">📈 Dataset Analytics Dashboard</span></div>')

                    with gr.Row():
                        dataset_stats = gr.HTML(
                            label="Dataset Statistics",
                            value="Upload a dataset to see statistics here...",
                            elem_classes=["white-text"]
                        )

                    with gr.Row():
                        with gr.Column():
                            viz_plot1 = gr.Plot(
                                label="📊 Dataset Analysis",
                                show_label=True,
                                elem_classes=["white-text"]
                            )
                        with gr.Column():
                            viz_plot2 = gr.Plot(
                                label="👥 HR Analytics",
                                show_label=True,
                                elem_classes=["white-text"]
                            )

                    # Event handler
                    train_btn.click(
                        fn=train_model,
                        inputs=[csv_upload],
                        outputs=[training_output, viz_plot1, viz_plot2, dataset_stats]
                    )

                # Candidate Evaluation Tab (enhanced with ML integration)
                with gr.Tab("🎯 Candidate Evaluation", elem_id="evaluation-tab"):
                    gr.HTML("""
                    <div class="info-box">
                        <h3 style="margin: 0 0 10px 0; color: white !important;">🎯 AI-Powered Candidate Search</h3>
                        <p style="margin: 0; color: white !important;">Search for candidates by education level and location preference. Get AI predictions on their hiring suitability and download comprehensive reports for suitable candidates.</p>
                    </div>
                    """)
                    with gr.Row():
                        with gr.Column(scale=1):
                            gr.HTML('<div class="section-header"><span style="color: white !important;">🔍 Search Criteria</span></div>')

                            # Get unique values from dataset for dropdown choices
                            education_choices = ["All", "Diploma", "Bachelors", "Masters", "PhD"]
                            location_choices = ["All", "Bangalore", "Chennai", "Delhi", "Hyderabad", "Pune", "Mumbai", "Remote", "International"]

                            education_level = gr.Dropdown(
                                label="Education Level",
                                choices=education_choices,
                                value="All",
                                info="Select education level to filter candidates",
                                elem_classes=["white-text"]
                            )
                            location_preference = gr.Dropdown(
                                label="Location Preference",
                                choices=location_choices,
                                value="All",
                                info="Select preferred work location",
                                elem_classes=["white-text"]
                            )

                            search_btn = gr.Button(
                                "🔍 Search Candidates with AI Prediction",
                                variant="primary",
                                size="lg",
                                elem_classes=["white-text"]
                            )

                            download_report_btn = gr.Button(
                                "📄 Download Suitable Candidates Report",
                                variant="secondary",
                                size="lg",
                                elem_classes=["white-text"]
                            )

                        with gr.Column(scale=2):
                            gr.HTML('<div class="section-header"><span style="color: white !important;">📋 AI Search Results</span></div>')

                            search_results = gr.Textbox(
                                label="Candidate Search Results with AI Predictions",
                                lines=20,
                                max_lines=30,
                                interactive=False,
                                placeholder="Search results with AI predictions will appear here...\n\nFormat: Candidate_ID | Email | AI Prediction (Suitable/Not Suitable) | Probability",
                                show_copy_button=True,
                                elem_classes=["white-text"]

                            )

                            # File download component
                            report_file = gr.File(
                                label="📄 Download Generated Report",
                                visible=True,
                                elem_classes=["white-text"]
                            )

                            report_status = gr.Textbox(
                                label="Report Generation Status",
                                lines=3,
                                interactive=False,
                                placeholder="Report generation status will appear here...",
                                elem_classes=["white-text"]
                            )

                    # Event handlers
                    search_btn.click(
                        fn=search_candidates_by_criteria,
                        inputs=[education_level, location_preference],
                        outputs=[search_results]
                    )

                    download_report_btn.click(
                        fn=download_suitable_candidates_report,
                        inputs=[],
                        outputs=[report_file, report_status]
                    )

                # User Guide Tab (fixed text visibility)
                with gr.Tab("📖 User Guide", elem_id="guide-tab"):
                    gr.HTML("""
                    <div style="padding: 20px; background: rgba(255, 255, 255, 0.95); border-radius: 15px; margin: 10px;">
                        <div class="info-box">
                            <h2 style="margin: 0; font-size: 2em; color: white !important;">📖 Complete User Guide</h2>
                            <p style="margin: 10px 0 0 0; opacity: 0.9; color: white !important;">Master the HR Prediction System with this comprehensive guide</p>
                        </div>

                        <div style="background: #ffffff; padding: 25px; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
                            <h3 style="color: #667eea !important; border-bottom: 2px solid #667eea; padding-bottom: 10px;">🚀 Getting Started</h3>

                            <div style="background: #f8f9fa; padding: 20px; border-radius: 10px; margin: 20px 0;">
                                <h4 style="color: #495057 !important;">Step 1: Train AI Models First</h4>
                                <ol style="color: #2c3e50 !important;">
                                    <li style="color: #2c3e50 !important;">Go to the <strong style="color: #2c3e50 !important;">AI Model Training</strong> tab</li>
                                    <li style="color: #2c3e50 !important;">Upload your HR dataset in CSV format with columns: Years_of_Experience, Education_Level, Skill_Set_Score, Job_Role_Level, Job_Type, Work_Mode, Location_Preference, Certifications_Count, Communication_Score, Problem_Solving_Score, Teamwork_Score, Leadership_Score, Cultural_Fit_Score, Assessment_Score, Expected_CTC, Notice_Period_Days, Hiring_Decision, Candidate_ID, E_Mail_ID</li>
                                    <li style="color: #2c3e50 !important;">Click <strong style="color: #2c3e50 !important;">"Train AI Models & Generate Analytics"</strong></li>
                                    <li style="color: #2c3e50 !important;">Review training results and model performance</li>
                                    <li style="color: #2c3e50 !important;">Use the trained model for candidate evaluation</li>
                                </ol>
                            </div>

                            <div style="background: #e8f4f8; padding: 20px; border-radius: 10px; margin: 20px 0;">
                                <h4 style="color: #495057 !important;">Step 2: Search Candidates with AI Predictions</h4>
                                <ol style="color: #2c3e50 !important;">
                                    <li style="color: #2c3e50 !important;">Go to the <strong style="color: #2c3e50 !important;">Candidate Evaluation</strong> tab</li>
                                    <li style="color: #2c3e50 !important;">Select education level and location preference filters</li>
                                    <li style="color: #2c3e50 !important;">Click <strong style="color: #2c3e50 !important;">"Search Candidates with AI Prediction"</strong></li>
                                    <li style="color: #2c3e50 !important;">View results showing Candidate_ID, Email, and AI hiring suitability prediction</li>
                                    <li style="color: #2c3e50 !important;">Download comprehensive report for suitable candidates</li>
                                </ol>
                            </div>
                        </div>

                        <div style="background: #ffffff; padding: 25px; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
                            <h3 style="color: #667eea !important; border-bottom: 2px solid #667eea; padding-bottom: 10px;">📊 Understanding the Results</h3>

                            <div style="background: #fff3cd; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 4px solid #ffc107;">
                                <h4 style="color: #856404 !important;">AI Model Training Results</h4>
                                <ul style="color: #2c3e50 !important;">
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Accuracy Score:</strong> Shows how well the model predicts hiring decisions</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Classification Report:</strong> Detailed performance metrics for hired/not hired predictions</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Feature Importance:</strong> Which candidate attributes matter most for hiring decisions</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Salary Prediction:</strong> R² score and mean absolute error for salary estimates</li>
                                </ul>
                            </div>

                            <div style="background: #d1ecf1; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 4px solid #17a2b8;">
                                <h4 style="color: #0c5460 !important;">Candidate Search Results</h4>
                                <ul style="color: #2c3e50 !important;">
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Candidate_ID:</strong> Unique identifier for each candidate from your dataset</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">E_Mail_ID:</strong> Contact information for candidates</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">AI Prediction:</strong> SUITABLE or NOT SUITABLE based on trained model</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Probability Score:</strong> Confidence level of the AI prediction (0.0-1.0)</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Success Rate:</strong> Percentage of candidates deemed suitable</li>
                                </ul>
                            </div>

                            <div style="background: #d4edda; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 4px solid #28a745;">
                                <h4 style="color: #155724 !important;">Report Generation</h4>
                                <ul style="color: #2c3e50 !important;">
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Suitable Candidates Report:</strong> PDF with full details of all suitable candidates</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Contact Information:</strong> Email addresses for direct outreach</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Detailed Profiles:</strong> Complete candidate information including scores and predictions</li>
                                    <li style="color: #2c3e50 !important;"><strong style="color: #2c3e50 !important;">Professional Format:</strong> Ready-to-share PDF reports for hiring teams</li>
                                </ul>
                            </div>
                        </div>

                        <div style="background: #ffffff; padding: 25px; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
                            <h3 style="color: #667eea !important; border-bottom: 2px solid #667eea; padding-bottom: 10px;">⚠️ Important Notes</h3>

                            <div style="background: #f8d7da; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 4px solid #dc3545;">
                                <h4 style="color: #721c24 !important;">Data Requirements</h4>
                                <ul style="color: #2c3e50 !important;">
                                    <li style="color: #2c3e50 !important;">Your CSV must contain all required columns for proper AI training</li>
                                    <li style="color: #2c3e50 !important;">Hiring_Decision column should contain clear Yes/No or Hired/Not Hired values</li>
                                    <li style="color: #2c3e50 !important;">Candidate_ID and E_Mail_ID columns are essential for search functionality</li>
                                    <li style="color: #2c3e50 !important;">Minimum 50+ records recommended for reliable AI predictions</li>
                                    <li style="color: #2c3e50 !important;">Missing values will be handled automatically but may affect accuracy</li>
                                </ul>
                            </div>

                            <div style="background: #e2e3e5; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 4px solid #6c757d;">
                                <h4 style="color: #495057 !important;">Best Practices</h4>
                                <ul style="color: #2c3e50 !important;">
                                    <li style="color: #2c3e50 !important;">Always train the model first before searching candidates</li>
                                    <li style="color: #2c3e50 !important;">Use specific education and location filters for targeted searches</li>
                                    <li style="color: #2c3e50 !important;">Review both AI predictions and candidate profiles before making decisions</li>
                                    <li style="color: #2c3e50 !important;">Download reports for suitable candidates to maintain records</li>
                                    <li style="color: #2c3e50 !important;">Consider probability scores when evaluating borderline candidates</li>
                                </ul>
                            </div>
                        </div>
                    </div>
                    """)

                # About Page Tab (fixed text visibility)
                with gr.Tab("ℹ️ About", elem_id="about-tab"):
                    gr.HTML("""
                    <div style="padding: 20px; background: rgba(255, 255, 255, 0.95); border-radius: 15px; margin: 10px;">
                        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 15px; color: white; margin-bottom: 30px; text-align: center;">
                            <h1 style="margin: 0; font-size: 2.5em; color: white !important;">ℹ️ About HR Prediction System</h1>
                            <p style="margin: 15px 0 0 0; font-size: 1.2em; opacity: 0.9; color: white !important;">Advanced AI-Powered Human Resources Management</p>
                        </div>

                        <div style="background: #ffffff; padding: 30px; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
                            <h2 style="color: #667eea !important; margin-top: 0;">🎯 Project Overview</h2>
                            <p style="color: #2c3e50 !important; font-size: 1.1em; line-height: 1.6;">
                                The HR Prediction System is a cutting-edge application that combines traditional rule-based hiring criteria
                                with advanced machine learning algorithms to provide comprehensive candidate evaluations. Built with Python,
                                XGBoost, and Gradio, it offers both automated screening and data-driven insights for modern HR departments.
                                This enhanced version includes candidate search functionality with AI predictions and comprehensive reporting.
                            </p>

                            <h3 style="color: #667eea !important;">✨ Key Features</h3>
                            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin: 20px 0;">
                                <div style="background: #f8f9fa; padding: 20px; border-radius: 10px;">
                                    <h4 style="color: #495057 !important; margin: 0 0 10px 0;">🤖 AI-Powered Candidate Search</h4>
                                    <ul style="color: #2c3e50 !important;">
                                        <li style="color: #2c3e50 !important;">Search candidates by education and location</li>
                                        <li style="color: #2c3e50 !important;">AI predictions for hiring suitability</li>
                                        <li style="color: #2c3e50 !important;">Probability scores for decision confidence</li>
                                        <li style="color: #2c3e50 !important;">Automatic filtering of suitable candidates</li>
                                    </ul>
                                </div>

                                <div style="background: #f8f9fa; padding: 20px; border-radius: 10px;">
                                    <h4 style="color: #495057 !important; margin: 0 0 10px 0;">📊 Advanced Analytics</h4>
                                    <ul style="color: #2c3e50 !important;">
                                        <li style="color: #2c3e50 !important;">Dataset visualization dashboard</li>
                                        <li style="color: #2c3e50 !important;">Model performance metrics</li>
                                        <li style="color: #2c3e50 !important;">Feature importance analysis</li>
                                        <li style="color: #2c3e50 !important;">HR analytics insights</li>
                                    </ul>
                                </div>

                                <div style="background: #f8f9fa; padding: 20px; border-radius: 10px;">
                                    <h4 style="color: #495057 !important; margin: 0 0 10px 0;">📄 Professional Reports</h4>
                                    <ul style="color: #2c3e50 !important;">
                                        <li style="color: #2c3e50 !important;">PDF reports for suitable candidates</li>
                                        <li style="color: #2c3e50 !important;">Complete candidate profiles</li>
                                        <li style="color: #2c3e50 !important;">Contact information included</li>
                                        <li style="color: #2c3e50 !important;">Professional formatting</li>
                                    </ul>
                                </div>

                                <div style="background: #f8f9fa; padding: 20px; border-radius: 10px;">
                                    <h4 style="color: #495057 !important; margin: 0 0 10px 0;">🔧 Machine Learning Models</h4>
                                    <ul style="color: #2c3e50 !important;">
                                        <li style="color: #2c3e50 !important;">XGBoost classification for hiring decisions</li>
                                        <li style="color: #2c3e50 !important;">Salary prediction regression models</li>
                                        <li style="color: #2c3e50 !important;">Automatic preprocessing and encoding</li>
                                        <li style="color: #2c3e50 !important;">Performance evaluation metrics</li>
                                    </ul>
                                </div>
                            </div>

                            <h3 style="color: #667eea !important;">🛠️ Technology Stack</h3>
                            <div style="background: #f8f9fa; padding: 20px; border-radius: 10px; margin: 20px 0;">
                                <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px;">
                                    <div style="text-align: center;">
                                        <h4 style="color: #495057 !important; margin: 0;">Backend</h4>
                                        <p style="color: #2c3e50 !important;">Python, Pandas, NumPy</p>
                                    </div>
                                    <div style="text-align: center;">
                                        <h4 style="color: #495057 !important; margin: 0;">Machine Learning</h4>
                                        <p style="color: #2c3e50 !important;">XGBoost, Scikit-learn</p>
                                    </div>
                                    <div style="text-align: center;">
                                        <h4 style="color: #495057 !important; margin: 0;">Visualization</h4>
                                        <p style="color: #2c3e50 !important;">Matplotlib, Seaborn, Plotly</p>
                                    </div>
                                    <div style="text-align: center;">
                                        <h4 style="color: #495057 !important; margin: 0;">Interface</h4>
                                        <p style="color: #2c3e50 !important;">Gradio, HTML/CSS</p>
                                    </div>
                                    <div style="text-align: center;">
                                        <h4 style="color: #495057 !important; margin: 0;">Reports</h4>
                                        <p style="color: #2c3e50 !important;">ReportLab PDF</p>
                                    </div>
                                </div>
                            </div>

                            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; color: white; text-align: center; margin: 30px 0;">
                                <h3 style="margin: 0 0 15px 0; color: white !important;">Enhanced Features</h3>
                                <p style="margin: 0; font-size: 1.1em; opacity: 0.9; color: white !important;">
                                    This enhanced version includes candidate search functionality, AI-powered hiring predictions,
                                    and comprehensive reporting capabilities. Perfect for modern HR departments looking to leverage
                                    data science for better hiring decisions.
                                </p>
                            </div>
                        </div>
                    </div>
                    """)

    return demo

# Main execution
if __name__ == "__main__":
    demo = create_interface()
    demo.launch(
        share=True,
        debug=True,
        show_error=True,
        inbrowser=True
    )

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a210e5370655085ec7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Loading and preprocessing data...
Successfully loaded file: /tmp/gradio/7ac082dfaaa6229304fd0d6d617d12ab2a6c74168db8cfdcf42e50d27d801ac4/IT_Hiring_Dataset.csv
Dataset shape: (200, 18)
Columns: ['Years_of_Experience', 'Education_Level', 'Skill_Set_Score', 'Job_Role_Level', 'Job_Type', 'Work_Mode', 'Location_Preference', 'Certifications_Count', 'Communication_Score', 'Problem_Solving_Score', 'Teamwork_Score', 'Leadership_Score', 'Cultural_Fit_Score', 'Assessment_Score', 'Expected_CTC', 'Notice_Period_Days', 'Hiring_Decision', 'Candidate_ID']
Encoded Education_Level: ['Bachelors', 'Diploma', 'Masters', 'PhD']
Encoded Job_Role_Level: ['Entry', 'Manager', 'Mid', 'Senior']
Encoded Job_Type: ['Contract', 'Full-time', 'Internship']
Encoded Work_Mode: ['Hybrid', 'On-site', 'Remote']
Encoded Location_Preference: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Pune', 'Remote']
Available features (15): ['Years_of_Experience', 'Education_Level_encoded', 'Skill_Set_Score', 'Job_Role_Level_encoded', 



---



---

# **Sentiment Analysis for the employee feedbacks**



---



---

